<a href="https://colab.research.google.com/github/oesquivel81/TopoRAG/blob/main/01_arxiv_corpus_collection_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TopoRAG — Notebook 01: arXiv Corpus Collection

This notebook builds the raw scientific corpus for **TopoRAG**, a
Retrieval-Augmented Generation system specialized in:

- Differential Geometry
- Algebraic Topology
- Homology and Cohomology
- Topological Data Analysis
- Persistent Homology

## Scope

This notebook is responsible only for corpus acquisition and raw page-level
text extraction. It does **not** perform text cleaning, chunking, embedding
generation, vector indexing, or RAG inference.

## Pipeline

arXiv queries  
→ metadata retrieval  
→ duplicate removal  
→ manifest creation  
→ PDF download  
→ page-level raw text extraction  
→ metadata enrichment  
→ structural validation  
→ Parquet export

## 1. Install collection dependencies

This step installs only the libraries required for corpus acquisition,
tabular storage, HTTP downloads, and PDF text extraction.

LangChain and LangGraph are intentionally excluded so that this notebook
remains independent from downstream RAG frameworks and avoids dependency
conflicts in Google Colab.

In [1]:
!pip install -q \
    arxiv \
    pymupdf \
    pandas \
    pyarrow \
    requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 39.5 MB/s eta 0:00:00


## 2. Import libraries and configure logging

The collection pipeline uses explicit standard-library and third-party
components rather than framework abstractions.

A notebook-level logger is configured once so repeated cell execution does
not produce duplicated handlers or duplicated log messages.

In [2]:
from __future__ import annotations

import json
import logging
import re
import time
from pathlib import Path
from typing import Any, Final, Iterable

import arxiv
import fitz
import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


LOGGER = logging.getLogger("toporag.arxiv_collection")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False

if not LOGGER.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter(
            fmt="%(asctime)s | %(levelname)s | %(message)s",
            datefmt="%H:%M:%S",
        )
    )
    LOGGER.addHandler(handler)

## 3. Configure project directories and output files

Raw PDFs, manifests, intermediate datasets, and processed datasets are kept
in separate directories.

Notebook 01 writes only to:

- `data/raw/arxiv`;
- `data/manifests`;
- `data/interim`.

The `data/processed` directory is created for project consistency, but it is
reserved for Notebook 02 and is not used as an output location here.

In [3]:
PROJECT_ROOT: Final[Path] = Path("/content/TopoRAG")

RAW_PDF_DIR: Final[Path] = PROJECT_ROOT / "data" / "raw" / "arxiv"
MANIFEST_DIR: Final[Path] = PROJECT_ROOT / "data" / "manifests"
INTERIM_DIR: Final[Path] = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR: Final[Path] = PROJECT_ROOT / "data" / "processed"

MANIFEST_CSV_PATH: Final[Path] = (
    MANIFEST_DIR / "arxiv_corpus_manifest.csv"
)
MANIFEST_PARQUET_PATH: Final[Path] = (
    MANIFEST_DIR / "arxiv_corpus_manifest.parquet"
)
RAW_PAGES_PARQUET_PATH: Final[Path] = (
    INTERIM_DIR / "arxiv_raw_pages.parquet"
)

for directory in (
    RAW_PDF_DIR,
    MANIFEST_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

LOGGER.info("Project root: %s", PROJECT_ROOT)
LOGGER.info("Raw PDF directory: %s", RAW_PDF_DIR)
LOGGER.info("Manifest directory: %s", MANIFEST_DIR)
LOGGER.info("Interim directory: %s", INTERIM_DIR)

20:18:43 | INFO | Project root: /content/TopoRAG
20:18:43 | INFO | Raw PDF directory: /content/TopoRAG/data/raw/arxiv
20:18:43 | INFO | Manifest directory: /content/TopoRAG/data/manifests
20:18:43 | INFO | Interim directory: /content/TopoRAG/data/interim


## 4. Define the domain-specific arXiv queries

Each query represents one semantic topic in the TopoRAG corpus.

Queries are executed independently so the collection process can retain
provenance. When the same article appears in multiple searches, it will later
be deduplicated by canonical `arxiv_id`, while all matching topics and search
queries will be preserved.

In [4]:
SEARCH_QUERIES: Final[dict[str, str]] = {
    "differential_geometry": (
        'cat:math.DG AND '
        '(all:"differential geometry" OR all:"Riemannian geometry")'
    ),
    "algebraic_topology": (
        'cat:math.AT AND '
        '(all:"algebraic topology" OR all:"homology" OR all:"cohomology")'
    ),
    "topological_data_analysis": (
        '(cat:math.AT OR cat:cs.LG OR cat:stat.ML) AND '
        '(all:"persistent homology" OR all:"topological data analysis")'
    ),
}

## 5. Configure one reusable arXiv client

A single `arxiv.Client` instance is reused for every search. This keeps
pagination, retry behavior, connection reuse, and request delays consistent
across topics.

`MAX_RESULTS_PER_QUERY` is intentionally isolated as a configuration value.
The default is suitable for validating the complete notebook in Colab before
scaling the corpus.

In [5]:
MAX_RESULTS_PER_QUERY: Final[int] = 10

client = arxiv.Client(
    page_size=10,
    delay_seconds=3,
    num_retries=3,
)

LOGGER.info(
    "Configured %d topic queries with up to %d results each.",
    len(SEARCH_QUERIES),
    MAX_RESULTS_PER_QUERY,
)

20:18:43 | INFO | Configured 3 topic queries with up to 10 results each.


## 6. Define identifier and metadata normalization helpers

arXiv results may include versioned identifiers such as `2501.01234v2`.
The corpus uses a canonical identifier without the version suffix for
deduplication and stable joins.

The original version is stored separately. Legacy arXiv identifiers may
contain `/`, so a filesystem-safe PDF filename is generated independently
without altering the canonical identifier stored in the manifest.

In [6]:
ARXIV_VERSION_PATTERN: Final[re.Pattern[str]] = re.compile(
    r"v(?P<version>\d+)$"
)


def split_arxiv_identifier(short_id: str) -> tuple[str, int | None]:
    """
    Split an arXiv short identifier into canonical ID and version number.

    Args:
        short_id:
            Versioned or unversioned arXiv short identifier.

    Returns:
        Canonical arXiv ID and optional version number.

    Raises:
        ValueError:
            If the supplied identifier is empty.
    """
    normalized = short_id.strip()

    if not normalized:
        raise ValueError("The arXiv identifier cannot be empty.")

    version_match = ARXIV_VERSION_PATTERN.search(normalized)

    if version_match is None:
        return normalized, None

    version = int(version_match.group("version"))
    canonical_id = ARXIV_VERSION_PATTERN.sub("", normalized)

    return canonical_id, version


def make_pdf_filename(arxiv_id: str) -> str:
    """
    Convert a canonical arXiv ID into a filesystem-safe PDF filename.

    Args:
        arxiv_id:
            Canonical arXiv identifier.

    Returns:
        Safe filename ending in `.pdf`.
    """
    safe_identifier = arxiv_id.replace("/", "__")
    return f"{safe_identifier}.pdf"


def normalize_metadata_text(value: str | None) -> str | None:
    """
    Trim metadata text without modifying mathematical page content.

    Args:
        value:
            Optional metadata string.

    Returns:
        Trimmed string or None.
    """
    if value is None:
        return None

    normalized = value.strip()
    return normalized or None


def result_to_hit_record(
    paper: arxiv.Result,
    topic: str,
    search_query: str,
) -> dict[str, Any]:
    """
    Convert one arXiv result into a serializable query-hit record.

    Args:
        paper:
            Result returned by the arXiv client.
        topic:
            TopoRAG topic that produced the result.
        search_query:
            Exact arXiv query used for retrieval.

    Returns:
        Metadata and search-provenance dictionary.
    """
    short_id = paper.get_short_id()
    arxiv_id, arxiv_version = split_arxiv_identifier(short_id)

    return {
        "arxiv_id": arxiv_id,
        "arxiv_version": arxiv_version,
        "entry_id": paper.entry_id,
        "pdf_url": paper.pdf_url,
        "title": normalize_metadata_text(paper.title),
        "authors": [
            author.name.strip()
            for author in paper.authors
            if author.name.strip()
        ],
        "published": paper.published,
        "updated": paper.updated,
        "primary_category": paper.primary_category,
        "categories": sorted(set(paper.categories)),
        "summary": normalize_metadata_text(paper.summary),
        "comment": normalize_metadata_text(paper.comment),
        "journal_ref": normalize_metadata_text(paper.journal_ref),
        "doi": normalize_metadata_text(paper.doi),
        "topic": topic,
        "search_query": search_query,
        "source_type": "arxiv",
        "retrieved_at_utc": pd.Timestamp.now(tz="UTC"),
    }

## 7. Retrieve metadata for every topic query

Each query is executed independently with the same client. Results are stored
as plain dictionaries instead of keeping live `arxiv.Result` objects, which
makes the collection state serializable and easier to validate.

A failure in one topic is recorded without discarding successful results from
the other topics. The notebook stops only when no metadata can be retrieved
at all.

In [7]:
def collect_arxiv_query_hits(
    arxiv_client: arxiv.Client,
    search_queries: dict[str, str],
    max_results_per_query: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Execute all configured arXiv queries and collect metadata records.

    Args:
        arxiv_client:
            Reusable arXiv API client.
        search_queries:
            Mapping from TopoRAG topic to arXiv query string.
        max_results_per_query:
            Maximum number of results retrieved for each query.

    Returns:
        Query-hit metadata and query-error DataFrames.

    Raises:
        ValueError:
            If max_results_per_query is not positive.
        RuntimeError:
            If no metadata records can be retrieved.
    """
    if max_results_per_query <= 0:
        raise ValueError("max_results_per_query must be positive.")

    hit_records: list[dict[str, Any]] = []
    error_records: list[dict[str, str]] = []

    for topic, query in search_queries.items():
        LOGGER.info("Searching topic: %s", topic)

        search = arxiv.Search(
            query=query,
            max_results=max_results_per_query,
            sort_by=arxiv.SortCriterion.Relevance,
            sort_order=arxiv.SortOrder.Descending,
        )

        topic_count = 0

        try:
            for paper in arxiv_client.results(search):
                hit_records.append(
                    result_to_hit_record(
                        paper=paper,
                        topic=topic,
                        search_query=query,
                    )
                )
                topic_count += 1

            LOGGER.info(
                "Retrieved %d records for topic: %s",
                topic_count,
                topic,
            )

        except Exception as exc:
            LOGGER.exception(
                "Metadata retrieval failed for topic: %s",
                topic,
            )
            error_records.append(
                {
                    "topic": topic,
                    "search_query": query,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }
            )

    hits_df = pd.DataFrame(hit_records)
    errors_df = pd.DataFrame(
        error_records,
        columns=[
            "topic",
            "search_query",
            "error_type",
            "error_message",
        ],
    )

    if hits_df.empty:
        raise RuntimeError(
            "No arXiv metadata was retrieved. "
            "Review the network connection and query syntax."
        )

    return hits_df, errors_df


query_hits_df, query_errors_df = collect_arxiv_query_hits(
    arxiv_client=client,
    search_queries=SEARCH_QUERIES,
    max_results_per_query=MAX_RESULTS_PER_QUERY,
)

LOGGER.info("Total query hits: %d", len(query_hits_df))
LOGGER.info(
    "Unique canonical arXiv IDs before manifest creation: %d",
    query_hits_df["arxiv_id"].nunique(),
)

display(
    query_hits_df[
        [
            "arxiv_id",
            "title",
            "topic",
            "primary_category",
            "published",
        ]
    ].head(10)
)

if not query_errors_df.empty:
    display(query_errors_df)

20:18:43 | INFO | Searching topic: differential_geometry
20:18:43 | INFO | Retrieved 10 records for topic: differential_geometry
20:18:43 | INFO | Searching topic: algebraic_topology
20:18:46 | INFO | Retrieved 10 records for topic: algebraic_topology
20:18:46 | INFO | Searching topic: topological_data_analysis
20:18:50 | INFO | Retrieved 10 records for topic: topological_data_analysis
20:18:50 | INFO | Total query hits: 30
20:18:50 | INFO | Unique canonical arXiv IDs before manifest creation: 30


,arxiv_id,title,topic,primary_category,published
0,math/0504358,Discrete differential geometry. Consistency as...,differential_geometry,math.DG,2005-04-18 11:16:37+00:00
1,2601.02325,"A translation of ""What is differential geometr...",differential_geometry,math.HO,2026-01-05 18:16:44+00:00
2,1707.07549,Bonnet's type theorems in the relative differe...,differential_geometry,math.DG,2017-07-21 06:00:33+00:00
3,math/0505221,Sasakian Geometry and Einstein Metrics on Spheres,differential_geometry,math.DG,2005-05-11 16:02:27+00:00
4,1206.3093,Sub-riemannian geometry from intrinsic viewpoint,differential_geometry,math.MG,2012-06-14 12:48:23+00:00
5,1807.11290,Riemannian geometry for shape analysis and com...,differential_geometry,math.DG,2018-07-30 11:19:28+00:00
6,2601.17166,Recovering Riemannian Geometry from Diffusion,differential_geometry,math.DG,2026-01-23 20:53:42+00:00
7,2408.05318,Recent Advances in Metallic Riemannian Geometr...,differential_geometry,math.DG,2024-08-09 19:43:21+00:00
8,2605.02279,Foundations of Riemannian Geometry for Riemann...,differential_geometry,math.DG,2026-05-04 07:11:11+00:00
9,1105.6294,"Stringy differential geometry, beyond Riemann",differential_geometry,hep-th,2011-05-31 14:18:49+00:00


## 8. Deduplicate articles and preserve all topic provenance

The query-hit table may contain the same paper more than once. Deduplication
uses the canonical `arxiv_id`, not the title or PDF URL.

For every unique article, the manifest preserves:

- every matching TopoRAG topic;
- every search query that returned the article;
- the number of topic matches;
- the number of total query hits.

The most recently updated metadata record is retained as the document-level
metadata source.

In [8]:
def sorted_unique_strings(values: Iterable[str]) -> list[str]:
    """
    Return unique non-empty strings in deterministic sorted order.

    Args:
        values:
            Iterable of strings.

    Returns:
        Sorted list of unique values.
    """
    return sorted(
        {
            value.strip()
            for value in values
            if isinstance(value, str) and value.strip()
        }
    )


def build_corpus_manifest(query_hits: pd.DataFrame) -> pd.DataFrame:
    """
    Build one manifest row per canonical arXiv ID.

    Args:
        query_hits:
            Metadata rows collected independently for each topic query.

    Returns:
        Deduplicated corpus manifest with aggregated provenance.

    Raises:
        ValueError:
            If required columns are missing or the input is empty.
    """
    required_columns = {
        "arxiv_id",
        "updated",
        "topic",
        "search_query",
    }
    missing_columns = required_columns.difference(query_hits.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required query-hit columns: {sorted(missing_columns)}"
        )

    if query_hits.empty:
        raise ValueError("Cannot build a manifest from an empty DataFrame.")

    latest_metadata = (
        query_hits
        .sort_values(
            by=["arxiv_id", "updated"],
            ascending=[True, True],
            na_position="first",
        )
        .drop_duplicates(subset=["arxiv_id"], keep="last")
        .drop(columns=["topic", "search_query"])
    )

    provenance = (
        query_hits
        .groupby("arxiv_id", as_index=False)
        .agg(
            topics=("topic", sorted_unique_strings),
            search_queries=("search_query", sorted_unique_strings),
            topic_count=(
                "topic",
                lambda values: len(sorted_unique_strings(values)),
            ),
            query_hit_count=("topic", "size"),
        )
    )

    manifest = latest_metadata.merge(
        provenance,
        on="arxiv_id",
        how="inner",
        validate="one_to_one",
    )

    manifest["pdf_filename"] = manifest["arxiv_id"].map(
        make_pdf_filename
    )
    manifest["pdf_relative_path"] = (
        "data/raw/arxiv/" + manifest["pdf_filename"]
    )

    manifest["download_status"] = "pending"
    manifest["download_error"] = None
    manifest["pdf_size_bytes"] = pd.Series(
        [pd.NA] * len(manifest),
        dtype="Int64",
    )
    manifest["downloaded_at_utc"] = pd.NaT

    manifest["extraction_status"] = "pending"
    manifest["extraction_error"] = None
    manifest["extracted_page_count"] = pd.Series(
        [pd.NA] * len(manifest),
        dtype="Int64",
    )

    manifest = (
        manifest
        .sort_values(
            by=["published", "arxiv_id"],
            ascending=[False, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    if manifest["arxiv_id"].duplicated().any():
        raise RuntimeError(
            "Manifest deduplication failed: duplicate arxiv_id values remain."
        )

    return manifest


manifest_df = build_corpus_manifest(query_hits_df)

LOGGER.info("Manifest documents: %d", len(manifest_df))
LOGGER.info(
    "Removed %d duplicate query hits.",
    len(query_hits_df) - len(manifest_df),
)

display(
    manifest_df[
        [
            "arxiv_id",
            "title",
            "topics",
            "topic_count",
            "query_hit_count",
            "pdf_filename",
        ]
    ].head(10)
)

20:18:50 | INFO | Manifest documents: 30
20:18:50 | INFO | Removed 0 duplicate query hits.


,arxiv_id,title,topics,topic_count,query_hit_count,pdf_filename
0,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],1,1,2605.02279.pdf
1,2603.13771,Brain Tumor Classification from 3D MRI Using P...,[topological_data_analysis],1,1,2603.13771.pdf
2,2601.17166,Recovering Riemannian Geometry from Diffusion,[differential_geometry],1,1,2601.17166.pdf
3,2601.02325,"A translation of ""What is differential geometr...",[differential_geometry],1,1,2601.02325.pdf
4,2408.05318,Recent Advances in Metallic Riemannian Geometr...,[differential_geometry],1,1,2408.05318.pdf
5,2305.04281,Analysing Multiscale Clusterings with Persiste...,[topological_data_analysis],1,1,2305.04281.pdf
6,2211.02520,"Magnitude, homology, and the Whitney twist",[algebraic_topology],1,1,2211.02520.pdf
7,2210.02569,"Semi-coarse Spaces, Homotopy and Homology",[algebraic_topology],1,1,2210.02569.pdf
8,2209.06155,On topological data analysis for SHM; an intro...,[topological_data_analysis],1,1,2209.06155.pdf
9,2209.05134,On topological data analysis for structural dy...,[topological_data_analysis],1,1,2209.05134.pdf


## 9. Export the initial corpus manifest

The manifest is written before PDF download so the selected corpus is
traceable even when a later network or extraction step fails.

The Parquet file preserves list-valued columns such as authors, categories,
topics, and search queries. The CSV representation serializes those columns
as JSON arrays so their structure remains unambiguous.

The same files are updated again at the end with download and extraction
statuses.

In [9]:
MANIFEST_LIST_COLUMNS: Final[tuple[str, ...]] = (
    "authors",
    "categories",
    "topics",
    "search_queries",
)


def export_manifest(
    manifest: pd.DataFrame,
    csv_path: Path,
    parquet_path: Path,
) -> None:
    """
    Export a corpus manifest to CSV and Parquet.

    Args:
        manifest:
            Deduplicated corpus manifest.
        csv_path:
            Destination CSV path.
        parquet_path:
            Destination Parquet path.
    """
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    manifest.to_parquet(
        parquet_path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    csv_manifest = manifest.copy()

    for column in MANIFEST_LIST_COLUMNS:
        if column in csv_manifest.columns:
            csv_manifest[column] = csv_manifest[column].map(
                lambda value: json.dumps(
                    value if isinstance(value, list) else [],
                    ensure_ascii=False,
                )
            )

    csv_manifest.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
    )


export_manifest(
    manifest=manifest_df,
    csv_path=MANIFEST_CSV_PATH,
    parquet_path=MANIFEST_PARQUET_PATH,
)

LOGGER.info("Initial CSV manifest: %s", MANIFEST_CSV_PATH)
LOGGER.info("Initial Parquet manifest: %s", MANIFEST_PARQUET_PATH)

20:18:50 | INFO | Initial CSV manifest: /content/TopoRAG/data/manifests/arxiv_corpus_manifest.csv
20:18:50 | INFO | Initial Parquet manifest: /content/TopoRAG/data/manifests/arxiv_corpus_manifest.parquet


## 10. Configure resilient PDF downloads

PDF downloads use a reusable HTTP session with retry and backoff behavior.
Files are first written to a temporary `.part` path and moved atomically only
after PDF validation succeeds.

A valid existing PDF is never downloaded again. If an existing file cannot
be opened as a PDF, it is preserved with an `.invalid` suffix before a fresh
download is attempted.

In [10]:
PDF_REQUEST_TIMEOUT_SECONDS: Final[tuple[int, int]] = (15, 180)
PDF_DOWNLOAD_DELAY_SECONDS: Final[float] = 1.0
DOWNLOAD_CHUNK_SIZE_BYTES: Final[int] = 1024 * 1024


def create_download_session() -> requests.Session:
    """
    Create an HTTP session with retry and exponential backoff.

    Returns:
        Configured requests Session.
    """
    retry_policy = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry_policy)

    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(
        {
            "User-Agent": (
                "TopoRAG/0.1 "
                "(research corpus collection; Python requests)"
            )
        }
    )

    return session


def validate_pdf_file(pdf_path: Path) -> tuple[bool, str | None]:
    """
    Verify that a local file can be opened as a non-empty PDF.

    Args:
        pdf_path:
            Path to the candidate PDF.

    Returns:
        File validity and optional error message.
    """
    if not pdf_path.exists():
        return False, "File does not exist."

    if pdf_path.stat().st_size == 0:
        return False, "File is empty."

    try:
        with fitz.open(str(pdf_path)) as document:
            if document.page_count <= 0:
                return False, "PDF contains no pages."

        return True, None

    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def next_invalid_backup_path(pdf_path: Path) -> Path:
    """
    Generate a non-conflicting backup path for an invalid existing PDF.

    Args:
        pdf_path:
            Original PDF path.

    Returns:
        Available backup path.
    """
    candidate = pdf_path.with_suffix(".invalid.pdf")
    counter = 1

    while candidate.exists():
        candidate = pdf_path.with_suffix(
            f".invalid.{counter}.pdf"
        )
        counter += 1

    return candidate


def download_pdf(
    session: requests.Session,
    arxiv_id: str,
    pdf_url: str | None,
    output_path: Path,
) -> dict[str, Any]:
    """
    Download one arXiv PDF unless a valid local copy already exists.

    Args:
        session:
            Reusable HTTP session.
        arxiv_id:
            Canonical document identifier.
        pdf_url:
            Source PDF URL.
        output_path:
            Final local PDF path.

    Returns:
        Download status record.
    """
    base_record: dict[str, Any] = {
        "arxiv_id": arxiv_id,
        "download_status": "failed",
        "download_error": None,
        "pdf_size_bytes": pd.NA,
        "downloaded_at_utc": pd.NaT,
    }

    if output_path.exists():
        is_valid, validation_error = validate_pdf_file(output_path)

        if is_valid:
            return {
                **base_record,
                "download_status": "existing",
                "pdf_size_bytes": output_path.stat().st_size,
            }

        backup_path = next_invalid_backup_path(output_path)
        output_path.replace(backup_path)
        LOGGER.warning(
            "Preserved invalid existing PDF as %s: %s",
            backup_path.name,
            validation_error,
        )

    if not pdf_url:
        return {
            **base_record,
            "download_status": "missing_pdf_url",
            "download_error": "arXiv metadata did not provide a PDF URL.",
        }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(".pdf.part")

    try:
        with session.get(
            pdf_url,
            stream=True,
            timeout=PDF_REQUEST_TIMEOUT_SECONDS,
        ) as response:
            response.raise_for_status()

            with temporary_path.open("wb") as output_file:
                for chunk in response.iter_content(
                    chunk_size=DOWNLOAD_CHUNK_SIZE_BYTES
                ):
                    if chunk:
                        output_file.write(chunk)

        is_valid, validation_error = validate_pdf_file(temporary_path)

        if not is_valid:
            raise ValueError(
                f"Downloaded file is not a valid PDF: {validation_error}"
            )

        temporary_path.replace(output_path)

        return {
            **base_record,
            "download_status": "downloaded",
            "pdf_size_bytes": output_path.stat().st_size,
            "downloaded_at_utc": pd.Timestamp.now(tz="UTC"),
        }

    except Exception as exc:
        temporary_path.unlink(missing_ok=True)

        return {
            **base_record,
            "download_status": "failed",
            "download_error": f"{type(exc).__name__}: {exc}",
        }

## 11. Download all manifest PDFs without duplicating existing files

The download loop uses the deduplicated manifest, so each canonical
`arxiv_id` is processed only once.

Download outcomes are joined back into the manifest by `arxiv_id`. A short
delay is applied only after a new successful download; existing PDFs do not
incur an unnecessary delay.

In [11]:
def download_manifest_pdfs(
    manifest: pd.DataFrame,
    raw_pdf_dir: Path,
) -> pd.DataFrame:
    """
    Download PDFs for every manifest row.

    Args:
        manifest:
            Deduplicated corpus manifest.
        raw_pdf_dir:
            Directory for raw arXiv PDFs.

    Returns:
        One download result per arXiv ID.
    """
    result_records: list[dict[str, Any]] = []
    session = create_download_session()

    try:
        for row in manifest.itertuples(index=False):
            output_path = raw_pdf_dir / row.pdf_filename

            result = download_pdf(
                session=session,
                arxiv_id=row.arxiv_id,
                pdf_url=row.pdf_url,
                output_path=output_path,
            )
            result_records.append(result)

            LOGGER.info(
                "PDF %s: %s",
                row.arxiv_id,
                result["download_status"],
            )

            if result["download_status"] == "downloaded":
                time.sleep(PDF_DOWNLOAD_DELAY_SECONDS)

    finally:
        session.close()

    return pd.DataFrame(result_records)


download_results_df = download_manifest_pdfs(
    manifest=manifest_df,
    raw_pdf_dir=RAW_PDF_DIR,
)

download_columns = [
    "arxiv_id",
    "download_status",
    "download_error",
    "pdf_size_bytes",
    "downloaded_at_utc",
]

manifest_df = (
    manifest_df
    .drop(columns=download_columns[1:])
    .merge(
        download_results_df[download_columns],
        on="arxiv_id",
        how="left",
        validate="one_to_one",
    )
)

download_summary_df = (
    manifest_df["download_status"]
    .value_counts(dropna=False)
    .rename_axis("download_status")
    .reset_index(name="document_count")
)

display(download_summary_df)

20:18:51 | INFO | PDF 2605.02279: downloaded
20:18:52 | INFO | PDF 2603.13771: downloaded
20:18:53 | INFO | PDF 2601.17166: downloaded
20:18:54 | INFO | PDF 2601.02325: downloaded
20:18:55 | INFO | PDF 2408.05318: downloaded
20:18:57 | INFO | PDF 2305.04281: downloaded
20:18:58 | INFO | PDF 2211.02520: downloaded
20:18:59 | INFO | PDF 2210.02569: downloaded
20:19:00 | INFO | PDF 2209.06155: downloaded
20:19:01 | INFO | PDF 2209.05134: downloaded
20:19:03 | INFO | PDF 2111.05663: downloaded
20:19:04 | INFO | PDF 2110.02458: downloaded
20:19:11 | INFO | PDF 2011.14688: downloaded
20:19:12 | INFO | PDF 1811.04757: downloaded
20:19:13 | INFO | PDF 1807.11290: downloaded
20:19:14 | INFO | PDF 1708.07390: downloaded
20:19:15 | INFO | PDF 1707.07549: downloaded
20:19:16 | INFO | PDF 1512.01700: downloaded
20:19:17 | INFO | PDF 1506.07123: downloaded
20:19:19 | INFO | PDF 1304.7846: downloaded
20:19:20 | INFO | PDF 1206.3093: downloaded
20:19:21 | INFO | PDF 1202.6132: downloaded
20:19:22 | IN

,download_status,document_count
0,downloaded,30


## 12. Extract raw text page by page

PyMuPDF opens each valid raw PDF and extracts text independently for every
page. The page text is not cleaned, normalized, chunked, or filtered in this
notebook.

Only lightweight structural measurements are added:

- page number;
- total page count;
- character count;
- whitespace-based word count;
- raw empty-page flag.

These fields support validation, while Notebook 02 remains responsible for
quality assessment and text normalization.

In [12]:
def extract_pdf_pages(
    pdf_path: Path,
    arxiv_id: str,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """
    Extract raw text from every page in one PDF.

    Args:
        pdf_path:
            Local raw PDF path.
        arxiv_id:
            Canonical document identifier.

    Returns:
        Page records and one document extraction status.
    """
    page_records: list[dict[str, Any]] = []

    status_record: dict[str, Any] = {
        "arxiv_id": arxiv_id,
        "extraction_status": "failed",
        "extraction_error": None,
        "extracted_page_count": pd.NA,
    }

    try:
        with fitz.open(str(pdf_path)) as document:
            total_pages = document.page_count

            if total_pages <= 0:
                raise ValueError("PDF contains no pages.")

            extraction_timestamp = pd.Timestamp.now(tz="UTC")

            for page_index in range(total_pages):
                page = document.load_page(page_index)
                raw_text = page.get_text("text")

                page_records.append(
                    {
                        "arxiv_id": arxiv_id,
                        "source_file": pdf_path.name,
                        "page_number": page_index + 1,
                        "total_pages": total_pages,
                        "page_content": raw_text,
                        "char_count": len(raw_text),
                        "word_count": len(raw_text.split()),
                        "is_empty_raw": not bool(raw_text.strip()),
                        "extracted_at_utc": extraction_timestamp,
                    }
                )

        status_record.update(
            {
                "extraction_status": "success",
                "extracted_page_count": len(page_records),
            }
        )

        return page_records, status_record

    except Exception as exc:
        status_record["extraction_error"] = (
            f"{type(exc).__name__}: {exc}"
        )
        return [], status_record


def extract_manifest_pages(
    manifest: pd.DataFrame,
    raw_pdf_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract pages from all successfully available manifest PDFs.

    Args:
        manifest:
            Manifest containing PDF filenames and download statuses.
        raw_pdf_dir:
            Directory containing raw PDFs.

    Returns:
        Raw page table and per-document extraction status table.
    """
    all_page_records: list[dict[str, Any]] = []
    extraction_records: list[dict[str, Any]] = []

    available_statuses = {"downloaded", "existing"}

    for row in manifest.itertuples(index=False):
        if row.download_status not in available_statuses:
            extraction_records.append(
                {
                    "arxiv_id": row.arxiv_id,
                    "extraction_status": "skipped",
                    "extraction_error": (
                        "PDF unavailable because download status is "
                        f"{row.download_status}."
                    ),
                    "extracted_page_count": pd.NA,
                }
            )
            continue

        pdf_path = raw_pdf_dir / row.pdf_filename

        page_records, extraction_status = extract_pdf_pages(
            pdf_path=pdf_path,
            arxiv_id=row.arxiv_id,
        )

        all_page_records.extend(page_records)
        extraction_records.append(extraction_status)

        LOGGER.info(
            "Extraction %s: %s (%s pages)",
            row.arxiv_id,
            extraction_status["extraction_status"],
            extraction_status["extracted_page_count"],
        )

    pages_df = pd.DataFrame(
        all_page_records,
        columns=[
            "arxiv_id",
            "source_file",
            "page_number",
            "total_pages",
            "page_content",
            "char_count",
            "word_count",
            "is_empty_raw",
            "extracted_at_utc",
        ],
    )

    extraction_df = pd.DataFrame(
        extraction_records,
        columns=[
            "arxiv_id",
            "extraction_status",
            "extraction_error",
            "extracted_page_count",
        ],
    )

    return pages_df, extraction_df


pages_df, extraction_results_df = extract_manifest_pages(
    manifest=manifest_df,
    raw_pdf_dir=RAW_PDF_DIR,
)

extraction_columns = [
    "arxiv_id",
    "extraction_status",
    "extraction_error",
    "extracted_page_count",
]

manifest_df = (
    manifest_df
    .drop(columns=extraction_columns[1:])
    .merge(
        extraction_results_df[extraction_columns],
        on="arxiv_id",
        how="left",
        validate="one_to_one",
    )
)

LOGGER.info("Raw pages extracted: %d", len(pages_df))

display(
    manifest_df["extraction_status"]
    .value_counts(dropna=False)
    .rename_axis("extraction_status")
    .reset_index(name="document_count")
)

20:19:32 | INFO | Extraction 2605.02279: success (143 pages)
20:19:32 | INFO | Extraction 2603.13771: success (21 pages)
20:19:32 | INFO | Extraction 2601.17166: success (20 pages)
20:19:33 | INFO | Extraction 2601.02325: success (214 pages)
20:19:33 | INFO | Extraction 2408.05318: success (47 pages)
20:19:34 | INFO | Extraction 2305.04281: success (22 pages)
20:19:34 | INFO | Extraction 2211.02520: success (25 pages)
20:19:34 | INFO | Extraction 2210.02569: success (56 pages)
20:19:34 | INFO | Extraction 2209.06155: success (18 pages)
20:19:34 | INFO | Extraction 2209.05134: success (25 pages)
20:19:34 | INFO | Extraction 2111.05663: success (16 pages)
20:19:34 | INFO | Extraction 2110.02458: success (16 pages)
20:19:34 | INFO | Extraction 2011.14688: success (14 pages)
20:19:35 | INFO | Extraction 1811.04757: success (30 pages)
20:19:35 | INFO | Extraction 1807.11290: success (20 pages)
20:19:35 | INFO | Extraction 1708.07390: success (33 pages)
20:19:35 | INFO | Extraction 1707.0754

,extraction_status,document_count
0,success,30


## 13. Join page text with document metadata

The page table is enriched using a many-to-one join from pages to the
deduplicated manifest.

This preserves document-level metadata and all topic provenance on every page
without duplicating document records before extraction. The join is validated
explicitly so an accidental duplicate `arxiv_id` in the manifest fails
immediately.

In [13]:
if pages_df.empty:
    raise RuntimeError(
        "No pages were extracted. Review download and extraction statuses."
    )

corpus_df = pages_df.merge(
    manifest_df,
    on="arxiv_id",
    how="left",
    validate="many_to_one",
)

preferred_page_columns = [
    "arxiv_id",
    "source_file",
    "page_number",
    "total_pages",
    "page_content",
    "char_count",
    "word_count",
    "is_empty_raw",
    "extracted_at_utc",
    "title",
    "authors",
    "published",
    "updated",
    "entry_id",
    "pdf_url",
    "primary_category",
    "categories",
    "summary",
    "comment",
    "journal_ref",
    "doi",
    "topics",
    "search_queries",
    "topic_count",
    "query_hit_count",
    "source_type",
    "pdf_filename",
    "pdf_relative_path",
    "download_status",
    "pdf_size_bytes",
    "downloaded_at_utc",
    "extraction_status",
    "extracted_page_count",
]

remaining_columns = [
    column
    for column in corpus_df.columns
    if column not in preferred_page_columns
]

corpus_df = corpus_df[
    preferred_page_columns + remaining_columns
]

display(
    corpus_df[
        [
            "arxiv_id",
            "title",
            "topics",
            "page_number",
            "total_pages",
            "word_count",
            "is_empty_raw",
        ]
    ].head(10)
)

,arxiv_id,title,topics,page_number,total_pages,word_count,is_empty_raw
0,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],1,143,338,False
1,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],2,143,2025,False
2,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],3,143,2309,False
3,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],4,143,2276,False
4,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],5,143,1132,False
5,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],6,143,823,False
6,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],7,143,703,False
7,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],8,143,607,False
8,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],9,143,795,False
9,2605.02279,Foundations of Riemannian Geometry for Riemann...,[differential_geometry],10,143,434,False


## 14. Validate structural corpus invariants

Notebook 01 performs structural validation rather than semantic text-quality
cleaning.

Errors stop the export when they would make the dataset unreliable, such as
duplicate document IDs, duplicate page keys, failed metadata joins, or invalid
page numbering.

Warnings are reported for recoverable conditions such as individual download
failures, extraction failures, or raw empty pages. Those records remain
traceable through the manifest.

In [14]:
def make_validation_record(
    check_name: str,
    severity: str,
    passed: bool,
    details: str,
) -> dict[str, Any]:
    """
    Create one standardized validation result.

    Args:
        check_name:
            Stable validation check name.
        severity:
            Either `error` or `warning`.
        passed:
            Whether the check passed.
        details:
            Human-readable diagnostic information.

    Returns:
        Validation result dictionary.
    """
    return {
        "check_name": check_name,
        "severity": severity,
        "passed": bool(passed),
        "details": details,
    }


def validate_collection_outputs(
    manifest: pd.DataFrame,
    pages: pd.DataFrame,
    corpus: pd.DataFrame,
) -> pd.DataFrame:
    """
    Validate manifest, raw pages, and enriched corpus structure.

    Args:
        manifest:
            Final document manifest.
        pages:
            Raw page extraction table before metadata enrichment.
        corpus:
            Page table enriched with document metadata.

    Returns:
        Validation outcomes.
    """
    validations: list[dict[str, Any]] = []

    required_manifest_columns = {
        "arxiv_id",
        "title",
        "entry_id",
        "topics",
        "pdf_filename",
        "download_status",
        "extraction_status",
    }

    validations.append(
        make_validation_record(
            check_name="manifest_not_empty",
            severity="error",
            passed=not manifest.empty,
            details=f"Manifest rows: {len(manifest)}",
        )
    )

    missing_manifest_columns = required_manifest_columns.difference(
        manifest.columns
    )
    validations.append(
        make_validation_record(
            check_name="manifest_required_columns",
            severity="error",
            passed=not missing_manifest_columns,
            details=f"Missing columns: {sorted(missing_manifest_columns)}",
        )
    )

    duplicate_id_count = int(
        manifest["arxiv_id"].duplicated().sum()
    )
    validations.append(
        make_validation_record(
            check_name="manifest_unique_arxiv_id",
            severity="error",
            passed=duplicate_id_count == 0,
            details=f"Duplicate IDs: {duplicate_id_count}",
        )
    )

    required_null_count = int(
        manifest[
            ["arxiv_id", "title", "entry_id"]
        ].isna().sum().sum()
    )
    validations.append(
        make_validation_record(
            check_name="required_metadata_complete",
            severity="error",
            passed=required_null_count == 0,
            details=(
                "Null required metadata cells: "
                f"{required_null_count}"
            ),
        )
    )

    topic_validity = manifest["topics"].map(
        lambda value: isinstance(value, list) and len(value) > 0
    )
    invalid_topic_count = int((~topic_validity).sum())
    validations.append(
        make_validation_record(
            check_name="topics_preserved",
            severity="error",
            passed=invalid_topic_count == 0,
            details=(
                "Documents without topic provenance: "
                f"{invalid_topic_count}"
            ),
        )
    )

    validations.append(
        make_validation_record(
            check_name="pages_not_empty",
            severity="error",
            passed=not pages.empty,
            details=f"Extracted page rows: {len(pages)}",
        )
    )

    duplicate_page_keys = int(
        pages.duplicated(
            subset=["arxiv_id", "page_number"]
        ).sum()
    )
    validations.append(
        make_validation_record(
            check_name="unique_document_page_keys",
            severity="error",
            passed=duplicate_page_keys == 0,
            details=f"Duplicate page keys: {duplicate_page_keys}",
        )
    )

    page_numbers_valid = bool(
        (
            (pages["page_number"] >= 1)
            & (pages["page_number"] <= pages["total_pages"])
        ).all()
    )
    validations.append(
        make_validation_record(
            check_name="page_numbers_in_range",
            severity="error",
            passed=page_numbers_valid,
            details="Every page number must be between 1 and total_pages.",
        )
    )

    joined_metadata_complete = (
        corpus["title"].notna().all()
        and corpus["topics"].notna().all()
    )
    validations.append(
        make_validation_record(
            check_name="page_metadata_join_complete",
            severity="error",
            passed=joined_metadata_complete,
            details=(
                "Pages missing titles: "
                f"{int(corpus['title'].isna().sum())}; "
                "pages missing topics: "
                f"{int(corpus['topics'].isna().sum())}"
            ),
        )
    )

    failed_download_count = int(
        (~manifest["download_status"].isin(
            ["downloaded", "existing"]
        )).sum()
    )
    validations.append(
        make_validation_record(
            check_name="all_pdfs_available",
            severity="warning",
            passed=failed_download_count == 0,
            details=(
                "Documents without available PDFs: "
                f"{failed_download_count}"
            ),
        )
    )

    failed_extraction_count = int(
        (manifest["extraction_status"] != "success").sum()
    )
    validations.append(
        make_validation_record(
            check_name="all_documents_extracted",
            severity="warning",
            passed=failed_extraction_count == 0,
            details=(
                "Documents not successfully extracted: "
                f"{failed_extraction_count}"
            ),
        )
    )

    empty_page_count = int(pages["is_empty_raw"].sum())
    validations.append(
        make_validation_record(
            check_name="no_raw_empty_pages",
            severity="warning",
            passed=empty_page_count == 0,
            details=(
                "Raw empty pages detected: "
                f"{empty_page_count}. Notebook 02 will assess quality."
            ),
        )
    )

    outputs_are_separated = (
        MANIFEST_CSV_PATH.parent == MANIFEST_DIR
        and MANIFEST_PARQUET_PATH.parent == MANIFEST_DIR
        and RAW_PAGES_PARQUET_PATH.parent == INTERIM_DIR
        and RAW_PAGES_PARQUET_PATH.parent != PROCESSED_DIR
    )
    validations.append(
        make_validation_record(
            check_name="output_directory_separation",
            severity="error",
            passed=outputs_are_separated,
            details=(
                "Manifests must remain in manifests and raw pages "
                "must remain in interim."
            ),
        )
    )

    return pd.DataFrame(validations)


validation_df = validate_collection_outputs(
    manifest=manifest_df,
    pages=pages_df,
    corpus=corpus_df,
)

display(validation_df)

failed_error_checks = validation_df[
    (validation_df["severity"] == "error")
    & (~validation_df["passed"])
]

if not failed_error_checks.empty:
    raise RuntimeError(
        "Structural validation failed:\n"
        + failed_error_checks[
            ["check_name", "details"]
        ].to_string(index=False)
    )

,check_name,severity,passed,details
0,manifest_not_empty,error,True,Manifest rows: 30
1,manifest_required_columns,error,True,Missing columns: []
2,manifest_unique_arxiv_id,error,True,Duplicate IDs: 0
3,required_metadata_complete,error,True,Null required metadata cells: 0
4,topics_preserved,error,True,Documents without topic provenance: 0
5,pages_not_empty,error,True,Extracted page rows: 1325
6,unique_document_page_keys,error,True,Duplicate page keys: 0
7,page_numbers_in_range,error,True,Every page number must be between 1 and total_...
8,page_metadata_join_complete,error,True,Pages missing titles: 0; pages missing topics: 0
9,all_pdfs_available,warning,True,Documents without available PDFs: 0


## 15. Export the final manifest and raw page-level Parquet dataset

The final export updates the manifest with PDF download and extraction
statuses, then writes the enriched raw page dataset to
`data/interim/arxiv_raw_pages.parquet`.

The page dataset is deliberately intermediate. Text cleaning, Unicode
normalization, mathematical-symbol preservation checks, repeated header and
footer removal, quality filtering, and stable page IDs belong to Notebook 02.

In [15]:
export_manifest(
    manifest=manifest_df,
    csv_path=MANIFEST_CSV_PATH,
    parquet_path=MANIFEST_PARQUET_PATH,
)

corpus_df.to_parquet(
    RAW_PAGES_PARQUET_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

LOGGER.info("Final CSV manifest: %s", MANIFEST_CSV_PATH)
LOGGER.info("Final Parquet manifest: %s", MANIFEST_PARQUET_PATH)
LOGGER.info("Raw page Parquet: %s", RAW_PAGES_PARQUET_PATH)

20:19:39 | INFO | Final CSV manifest: /content/TopoRAG/data/manifests/arxiv_corpus_manifest.csv
20:19:39 | INFO | Final Parquet manifest: /content/TopoRAG/data/manifests/arxiv_corpus_manifest.parquet
20:19:39 | INFO | Raw page Parquet: /content/TopoRAG/data/interim/arxiv_raw_pages.parquet


## 16. Display the collection summary

The final summary confirms the number of query hits, unique papers,
downloaded or reused PDFs, extracted pages, empty raw pages, and generated
artifacts.

This is an operational summary only. Corpus-quality decisions remain deferred
to Notebook 02.

In [16]:
summary_df = pd.DataFrame(
    [
        {
            "metric": "query_hits",
            "value": len(query_hits_df),
        },
        {
            "metric": "unique_documents",
            "value": len(manifest_df),
        },
        {
            "metric": "duplicate_hits_removed",
            "value": len(query_hits_df) - len(manifest_df),
        },
        {
            "metric": "available_pdfs",
            "value": int(
                manifest_df["download_status"]
                .isin(["downloaded", "existing"])
                .sum()
            ),
        },
        {
            "metric": "successfully_extracted_documents",
            "value": int(
                (manifest_df["extraction_status"] == "success").sum()
            ),
        },
        {
            "metric": "raw_pages",
            "value": len(corpus_df),
        },
        {
            "metric": "raw_empty_pages",
            "value": int(corpus_df["is_empty_raw"].sum()),
        },
    ]
)

display(summary_df)

print("\nGenerated artifacts:")
print(f"- {MANIFEST_CSV_PATH}")
print(f"- {MANIFEST_PARQUET_PATH}")
print(f"- {RAW_PDF_DIR}/*.pdf")
print(f"- {RAW_PAGES_PARQUET_PATH}")

,metric,value
0,query_hits,30
1,unique_documents,30
2,duplicate_hits_removed,0
3,available_pdfs,30
4,successfully_extracted_documents,30
5,raw_pages,1325
6,raw_empty_pages,0



Generated artifacts:
- /content/TopoRAG/data/manifests/arxiv_corpus_manifest.csv
- /content/TopoRAG/data/manifests/arxiv_corpus_manifest.parquet
- /content/TopoRAG/data/raw/arxiv/*.pdf
- /content/TopoRAG/data/interim/arxiv_raw_pages.parquet
